In [ ]:
from pyspark.sql import SparkSession
import zipfile
import io
import boto3

# Inisialisasi Spark session
spark = SparkSession.builder \
    .appName("ExtractZipS3") \
    .getOrCreate()

# Inisialisasi S3 client
s3 = boto3.client(
    's3',
    region_name='ap-southeast-3',
    endpoint_url='https://s3.ap-southeast-3.amazonaws.com'
)

def extract_zip_s3_to_s3(source_bucket, zip_key, target_bucket, target_prefix):
    """
    Extract a ZIP file from one S3 location and upload the extracted files to another S3 location using PySpark.
    """
    # Ambil file ZIP dari S3 dan simpan ke dalam buffer
    response = s3.get_object(Bucket=source_bucket, Key=zip_key)
    zip_buffer = io.BytesIO(response['Body'].read())

    # Buka file ZIP dari buffer
    with zipfile.ZipFile(zip_buffer, 'r') as zip_ref:
        files_to_upload = []
        
        # Iterasi setiap file dalam ZIP
        for file_name in zip_ref.namelist():
            file_data = zip_ref.read(file_name)

            # Tambahkan data ke list untuk di-upload dengan parallel processing
            files_to_upload.append((file_name, file_data))

    # Fungsi upload untuk setiap file
    def upload_to_s3(file_tuple):
        file_name, file_data = file_tuple
        target_key = f"{target_prefix}/{file_name}"
        s3.put_object(Bucket=target_bucket, Key=target_key, Body=file_data)
        print(f"Uploaded {file_name} to s3://{target_bucket}/{target_key}")

    # Upload menggunakan PySpark parallel processing
    rdd = spark.sparkContext.parallelize(files_to_upload)
    rdd.foreach(upload_to_s3)

# Contoh penggunaan
source_bucket = 'bucket-pkb'
zip_key = 'input/yelp_academic_dataset_review.json.zip'
target_bucket = 'bucket-pkb'
target_prefix = 'output'

extract_zip_s3_to_s3(source_bucket, zip_key, target_bucket, target_prefix)


In [ ]:
from pyspark.sql import SparkSession
import zipfile
import io
import boto3
import botocore
import logging

# Inisialisasi Spark session dengan konfigurasi memori dan logging yang optimal
spark = SparkSession.builder \
    .appName("ExtractZipS3") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")  # Mengurangi log berlebih

# Inisialisasi S3 client dengan timeout dan retry mechanism
s3 = boto3.client(
    's3',
    region_name='ap-southeast-3',
    endpoint_url='https://s3.ap-southeast-3.amazonaws.com',
    config=botocore.config.Config(connect_timeout=60, retries={'max_attempts': 5})
)

def extract_and_move_zip_s3(source_bucket, zip_key, target_bucket, target_prefix):
    """
    Extract a ZIP file from one S3 location, upload the extracted files to another location, 
    and move the original ZIP file to the target prefix.
    """
    # Ambil file ZIP dari S3 dan simpan secara streaming ke dalam buffer
    try:
        response = s3.get_object(Bucket=source_bucket, Key=zip_key)
        zip_buffer = io.BytesIO(response['Body'].read())
    except Exception as e:
        logging.error(f"Error fetching ZIP from S3: {e}")
        return

    # Fungsi untuk meng-upload setiap file
    def upload_to_s3(file_tuple):
        file_name, file_data = file_tuple
        target_key = f"{target_prefix}/{file_name}"

        # Retry upload jika gagal
        for attempt in range(3):
            try:
                s3.put_object(Bucket=target_bucket, Key=target_key, Body=file_data)
                print(f"Uploaded {file_name} to s3://{target_bucket}/{target_key}")
                break
            except Exception as e:
                logging.warning(f"Retry {attempt + 1} for {file_name} failed: {e}")

    # Membaca dan upload file dalam batch untuk mengurangi beban memori
    with zipfile.ZipFile(zip_buffer, 'r') as zip_ref:
        file_names = zip_ref.namelist()

        # Fungsi untuk memproses batch file
        def process_batch(start, end):
            batch = [(name, zip_ref.read(name)) for name in file_names[start:end]]
            rdd = spark.sparkContext.parallelize(batch)
            rdd.foreach(upload_to_s3)

        # Batch upload, misalnya 10 file per batch
        batch_size = 10
        for i in range(0, len(file_names), batch_size):
            process_batch(i, i + batch_size)

    # Setelah ekstraksi, pindahkan ZIP file dari input ke output folder
    move_key = f"{target_prefix}/{zip_key.split('/')[-1]}"

    try:
        # Copy ZIP file ke lokasi baru
        s3.copy_object(
            Bucket=target_bucket,
            CopySource={'Bucket': source_bucket, 'Key': zip_key},
            Key=move_key
        )
        print(f"Copied {zip_key} to s3://{target_bucket}/{move_key}")

        # Hapus ZIP file dari lokasi asli
        s3.delete_object(Bucket=source_bucket, Key=zip_key)
        print(f"Deleted original ZIP file: s3://{source_bucket}/{zip_key}")
    except Exception as e:
        logging.error(f"Error moving ZIP file: {e}")

# Contoh penggunaan
source_bucket = 'bucket-pkb'
zip_key = 'input/yelp_academic_dataset_review.json.zip'
target_bucket = 'bucket-pkb'
target_prefix = 'output'

extract_and_move_zip_s3(source_bucket, zip_key, target_bucket, target_prefix)
